> **Standalone pipeline（2026-10-18）**：本 Notebook 的代码现由本目录的
> `pipeline/` 提供，无需 `../Jev-Mem` 克隆。默认 `JEV_MEM_MODE=offline`：
> 哈希词汇向量、启发式决策、抽取式回答、lexical F1，仅验证执行流程。
> 真 API 请设置 `JEV_MEM_MODE=live`，详见 [PIPELINE.md](PIPELINE.md)。
> 下文论文介绍、原上游路径、原始实验表格与解读保留为历史研究笔记，
> 不表示本实现的新实测结果。所有旧代码输出已清除；新结果以 run 的 summary.json 为准。

# Jev-Mem 缩放实验：对话变长时，四种做法各自付出什么代价

上一本 notebook（`jev_mem_walkthrough.ipynb`）在 16 轮对话上比了四个臂。这一本把问题换成**长程**：同一组 10 个目标事实、同一组 10 道题，把干扰对话从 **48 轮拉到 96、192 再到 384 轮**（4/8/16/32 个会话，跨 14 个多月），看每个臂的**质量、延迟、prompt tokens** 怎么缩放，算清**时间和 token 的损失对比**。另附真 LoCoMo 样本（300+ 轮英文对话）的 Jev-Mem 实测。最后回答一个问题：**为什么全量上下文准确度这么高，它的高是真本事还是假优势**。

实验设计要点（控制变量）：

- **题目不变**：10 道题在三个规模下完全相同，分数差异全部来自"干扰对话变多"；
- **事实不变**：10 个目标事实按相同比例位置植入（第 i 个事实在对话的 (i+0.5)/10 处），所以短对话里它们挨得近，长对话里散得开；
- **干扰递增**：填充闲聊 + 定向干扰句（别人的过敏、别人的 PMP、别人的燃油车），规模越大干扰越多；
- 全部真 API 实测：真 Jev 决策、DeepSeek 答题+评审、硅基流动 Qwen3-Embedding。

In [ ]:
import os, sys, json
from pathlib import Path

# 本地 standalone 包；无需上游克隆，也不改变工作目录。
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "Jev_mem" / "pipeline" / "memory.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从 LLM-Notebooks 根目录或 Jev_mem/ 启动 Notebook")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from Jev_mem.pipeline.notebooks import make_session
from Jev_mem.pipeline.data import walkthrough_dataset, synthetic_dataset
from Jev_mem.pipeline.memory import Memory

MODE = os.getenv("JEV_MEM_MODE", "offline")
OUTPUT = Path(os.getenv("JEV_MEM_OUTPUT", str(ROOT / "Jev_mem" / "artifacts")))
print("Implementation: standalone | Mode:", MODE)


In [ ]:
session = make_session(MODE)
print("embedding:", session.encoder.fingerprint)
print("decisions:", session.decisions.source)


## 1. 长程对话生成器：固定事实与题目，缩放干扰

10 个目标事实（过敏、健身、套餐、充电桩、尿酸、PMP、咖啡店、私教、姐姐买房、公积金）按等比位置植入；填充对话来自闲聊池，其中混入**定向干扰句**——问"用户对海鲜过敏吗"时对话里躺着"同事小李对虾过敏"，问"燃油车"时躺着"表哥的燃油车限行"。这些干扰专门考验检索臂会不会抓错证据。

In [ ]:
from Jev_mem.pipeline.data import synthetic_dataset
SCALES = [48, 96, 192, 384]
DATASETS = {n: synthetic_dataset(n) for n in SCALES}
for n, data in DATASETS.items():
    stamps = [o["timestamp"] for o in data["observations"]]
    assert stamps == sorted(stamps)
    print(n, "turns:", len(data["observations"]), "questions:", len(data["questions"]))


## 2. 四个臂（与走读实验同款）

A 滑动窗口（最近 6 条）· B 全上下文 · C 朴素 RAG（向量 Top-3）· D Jev-Mem。每个规模独立建一份 D 记忆，量出**建图时间随规模的增长**。

In [ ]:
# 四臂实现、建图、检索、使用量归属、评审都由共享 pipeline 提供。
# 每个规模创建独立的 run 目录，不复用旧图或审计日志。
from Jev_mem.pipeline.experiment import run_experiment


In [ ]:
RUNS = {}
for n in SCALES:
    RUNS[n] = session.run(DATASETS[n], OUTPUT)
    print(n, RUNS[n])


In [ ]:
SUMMARIES = {n: json.loads((run / "summary.json").read_text()) for n, run in RUNS.items()}
RAW = {n: [json.loads(line) for line in (run / "evaluations.jsonl").read_text().splitlines()]
       for n, run in RUNS.items()}
print("Metric:", SUMMARIES[SCALES[0]]["score_metric"])


In [ ]:
print("turns arm score prompt_tokens latency_seconds scored/total")
for n, summary in SUMMARIES.items():
    for arm, values in summary["arms"].items():
        print(n, arm, values["mean_score"], values["mean_prompt_tokens"],
              values["mean_answer_latency_seconds"], f"{values['scored']}/{values['questions']}")


In [ ]:
for n, run in RUNS.items():
    print(n, "SVG report:", run / "report.svg")


## 3. 为什么全量准确度这么高？——它高，是因为还没到它的失效区

先看实测衰减曲线（评审分）：

| 轮数 | A 滑窗 | B 全上下文 | C 朴素RAG | D Jev-Mem |
|---|---|---|---|---|
| 48 | 0.20 | **0.98** | 0.98 | 0.98 |
| 96 | 0.20 | **0.98** | 0.98 | 0.92 |
| 192 | 0.20 | **0.98** | 0.94 | 0.92 |
| 384 | 0.20 | 0.88 | 0.78 | **0.90** |

B 在 48~192 轮几乎满分，384 轮开始掉（0.88），在真 LoCoMo（419 轮、约 2 万 token 上下文）掉到 **0.60**。它短中期为什么高，四层原因按重要性排：

**1. 没有检索环节，就没有"没找到"这种失败。** RAG 类的失分模式是"该召回的没召回"或"召回了干扰"——干扰句每会话一条，越长的对话对检索臂越难（C 从 0.98 掉到 0.78 就是这么来的）。B 把全部证据摊在模型面前，只要装得下，信息可见性 100%，检索型问题对它没有难度曲线，只有成本曲线。

**2. 短中期远没到注意力稀释区。** deepseek-chat 窗口 128k token，192 轮才 ~5k token（占窗口 4%），"大海捞针"在这个密度下不难。384 轮（~10k token、16 条定向干扰）后 B 开始丢分——主要是时间题日期算错和细节张冠李戴，这正是长上下文注意力稀释的典型症状。

**3. 对抗压力小、judge 宽容。** 我们只有 2 道对抗题且参考答案就是"记忆中没有"，说"没有"就满分；论文 LoCoMo 的 Full Context 在 adversarial 类目只有 0.205，因为那些题专门诱导模型从相似记忆里**编造**。B 的高分有一部分是"这次没考到它的真实弱点"。

**4. judge 与答案模型同源。** DeepSeek 出答案 DeepSeek 评，风格匹配，对证据最全的 B 最有利。

**所以完整的答案是：B 的高分 = 装得下 + 没考弱点。它的失效区由上下文长度和干扰密度决定**——真 LoCoMo 上（下一节）B 0.60 对 D 0.98，质量优势在长程下不仅消失，还反转成劣势。

## 3.5 真 LoCoMo 长程实测：419 轮英文对话

以上是合成对话。再用数据集 [LoCoMo](https://github.com/snap-research/locomo) 的真实样本 0（**419 轮、74k 字符 ≈ 19.8k token**，Dyan/Caroline/Melanie 三人多会话）跑了一轮真刀真枪的对照（2026-09-25，真 Jev + DeepSeek，仓库官方基准 CLI + 同题全上下文脚本）：

| 指标 | B 全上下文 | D Jev-Mem |
|---|---|---|
| 每题 prompt tokens | **19,775** | **~784**（证据 15 节点实测） |
| LLM-as-a-Judge（同题 10 题） | 0.60 | **0.98** |
| 按题型 | 时间题 0.2~0.6 成片丢分 | 时间题全部满分 |
| 每题延迟（本机） | 0.60s | 3.83s（含 Jev 决策链 + judge） |
| 建图成本 | 0 | 494s（419 条观察，一次性） |

D 的记忆图：419 节点、9,462 条边（ENTITY 4090 / TEMPORAL 3726 / SEMANTIC 1502 / CAUSAL 144），真 Jev 决策，仅 3 次兜底。

三个要点：

1. **质量反转在真实数据上成立**：约 2 万 token 的上下文里，全上下文的时间题（"Caroline 什么时候去支持小组"）大面积丢分——模型面对 74k 字符算不准日期；Jev-Mem 沿 temporal 图直接召回正确证据，10 题里 9 题 1.0 分。这与论文 Full Context 0.481 的方向完全一致。
2. **token 剪刀差 25 倍**：19,775 对 784。问得越多省得越多，且 D 的证据 prompt 大小不随对话长度变化。
3. **诚实声明**：10 题是小样本，judge 是仓库自带的宽松量表（同期 F1 只有 57.3%，说明逐词匹配并不完美）；这组数字用于量级判断，不构成对论文 0.777 的复现（那是 1,986 题全量 + 更严协议）。

In [ ]:
for n, summary in SUMMARIES.items():
    b = summary["arms"]["B"]["mean_prompt_tokens"]
    d = summary["arms"]["D"]["mean_prompt_tokens"]
    ratio = b / d if b is not None and d is not None and d > 0 else None
    print(n, "B/D prompt token ratio:", ratio, "build_seconds:", summary["build_seconds"])
session.close()


## 4. 时间与 token 损失对比：怎么读这笔账

把四个臂放进"建图成本 + 每题成本 × 题数"的账本里（数字以上面实测为准）：

| 成本项 | A 滑窗 | B 全上下文 | C 朴素 RAG | D Jev-Mem |
|---|---|---|---|---|
| 建图（一次性） | 0 | 0 | 0（现场算向量，毫秒级） | **随轮数线性涨**（真 Jev 决策，每条观察约 0.5-0.7s） |
| 每题 token | 恒定但证据残缺 | **随轮数线性涨**（这是 B 的核心代价） | 恒定（Top-3 固定） | 恒定（只带筛选后的证据） |
| 每题延迟 | 最低档 | 随规模涨（prefill 变长） | 低 | 最高档（Jev 决策链 + embedding 往返） |
| 质量 | 最早崩（窗口外的全丢） | 短中期最高，超长才稀释 | 中，时间题必挂 | 中高，时间/多跳稳 |

三条实测结论：

1. **token 剪刀差是 D 的主场**：B 每题 token 随轮数线性涨，D 几乎水平——384 轮时 B/D 比值（上面速览实测）就是"每问一题省下的输入成本倍数"。对话越长、问得越多，这笔省得越多。
2. **时间账在本地 API 场景下 D 不占优**：D 的每题延迟比 B 高（Jev 决策是真金白银的时间），建图还要一次性投入。论文里 Jev-Mem 延迟最低（0.93s）比较对象是**其他记忆系统**（1.47~32.68s）和超长上下文（1.74s 且还在涨）——当 B 的 prefill 长到几万 token，B 的延迟线才会反超。**D 买的是 token 经济性和可审计性，不是本地的低延迟。**
3. **全上下文的高分是"还没到考验"**：质量上 B 会在对话长到注意力稀释时掉分（论文 LoCoMo 0.481），token 上它从第一轮就开始付线性税。选型的实际判据：**对话短（几百轮内）、问得少 → 直接 B；对话长、要反复问、要省钱省审计 → 上记忆系统，且值得为时间题上 Jev 决策面。**

## 5. 结论总览：三组实验连起来看

结合走读实验（`jev_mem_walkthrough.ipynb`，16 轮）、本篇缩放实验（48~384 轮）与真 LoCoMo（419 轮），所有实测结论收敛成一张矩阵：

| 实验 | 规模 | 一句话结论 |
|---|---|---|
| 走读四臂 | 16 轮 | 短对话 B 天花板（0.96）；D 0.84 > C 0.72，增益全在时间题 |
| 本篇缩放 | 48→384 轮 | B 384 轮掉到 0.88；D 最稳 0.90 反超第一；token 剪刀差 81× |
| 真 LoCoMo | 419 轮 / 19.8k tok | B 0.60 对 D 0.98，质量反转实锤；token 剪刀差 25× |

**七条有趣结论**

1. **决策面的增益与 embedding 质量正交。** 换更强的 embedding（MiniLM→Qwen3）把 C 拉高 0.08，但时间题 C 始终 0 分——"按时间线找证据"不靠语义相似度，这是 Jev 路由 + temporal 图视图的专属领地。
2. **token 是 D 的主场。** D 的证据 prompt 恒定（中文 ~120 tok、英文 ~784 tok），不随对话长度变化；B 线性上涨，384 轮 81 倍、真 LoCoMo 25 倍。问得越频繁、对话越长，省得越多。
3. **时间是 D 的成本项。** 建图线性（合成对话 ~0.45s/条，LoCoMo 长文本 ~1.2s/条），每题查询比 B 慢 1~3s（Jev 决策链）。本地 API 场景 D 不赢时间——它买的是 token 经济性、质量稳定性与可审计性。
4. **全量上下文的高分 = 装得下 + 没考弱点。** 失效区由上下文长度和干扰密度决定：≤192 轮近乎满分，384 轮开始稀释，2 万 token（LoCoMo）掉到 0.60。
5. **定向干扰是纯向量检索的毒药，对决策面伤害有限。** "同事对虾过敏"这类陷阱把 C 从 0.98 拖到 0.78，D 只从 0.98 掉到 0.90——图视图 + 路由比裸相似度抗干扰。
6. **建图经济学成立。** 384 条观察 = 767 次 Jev 决策调用（LoCoMo 419 条 ≈ 830 次）。按 TypeSafe 牌价（2026-09：输入 $0.042/百万 token、输出免费）粗估每次调用 ~1.5k 输入 token，整份 384 轮记忆的决策成本约 **5 美分**，一次性投入；对照论文里大模型记忆系统动辄上千秒、按生成计费的建图，这是量级差。
7. **答案长度会吃掉 judge 分数。** 仓库 System-Two 提示词要求"5-6 词简答"，中文多跳题漏细节被扣分（"改到下周三，推迟了"没提重新订票）。属于提示词工程问题，放宽答题提示即可回收，不是架构缺陷。

**选型判据**

- 对话几百轮以内、偶尔问答 → **直接全上下文**，别过度工程；
- 长对话、反复问答、在意 token 账单 → **上记忆系统**；
- 问题里时间线/因果/跨会话占比高 → **值得为 Jev 决策面付费**（D 对 C 的全部增益都在这类题上）；
- 本地低延迟场景 → embedding 本地化（省 API 往返），并接受建图的一次性时间投入。

**复现备忘（踩过的坑）**

- 仓库 `memory/llm_judge.py` 硬编码 judge 模型 `gpt-4o-mini`；配 DeepSeek 需 monkeypatch 模型名为 `deepseek-chat`（本 notebook 实验用的是自研 judge cell，无此问题）；
- 官方基准 CLI 必须 `--no-parallel`：macOS spawn 模式下并行 worker 不会继承主进程的 monkeypatch，静默失败；
- 记忆图缓存可复用（换配置才需 `--rebuild`），重建 LoCoMo 样本 0 约需 8 分钟；
- judge 分数轮间漂移 ±0.02~0.03，结论以同轮表格为准；本文所有分数为 LLM 评审的相对值，不构成对论文 LoCoMo 全量协议（1,986 题）的复现。